# LMF+KL Learning-Rate Ablation

This notebook mirrors the structure of `Ablations.ipynb`, but targets the extension method `lmf_KL`.

Protocol from Section 4.1 and Appendix E:

- unlearn each CoT step independently;
- use sentence-level steps, content-word/POS filtering, and FF2-only updates;
- run 5 unlearning epochs;
- use 30 pilot instances for learning-rate selection;
- hold out 20 instances for specificity;
- select the learning rate with maximum efficacy subject to `round(specificity) >= 95`.

`faithfulness` is reported for information only, matching Appendix E; it is not used to select the learning rate.

In [ ]:
import json
import math
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

METHOD = "lmf_KL"
RESULTS_DIR = "lmf_ablation"
N_UNLEARN = 30
N_VERIFY = 20
EPOCHS = 5
RT_LAMBDA = 1.0
SPECIFICITY_THRESHOLD = 95

# Compact grid for the 2x2 project subset. It brackets the NPO+KL best LR
# while keeping the pilot affordable.
COMPACT_LR_GRID = {
    ("Phi-3", "openbook"): [3e-5, 1e-4, 3e-4],
    ("Phi-3", "sqa"): [1e-5, 5e-5, 1e-4],
    ("LLaMA-3-3B", "openbook"): [1e-5, 3e-5, 5e-5],
    ("LLaMA-3-3B", "sqa"): [1e-5, 3e-5, 5e-5],
}

# Use this if the compact grid has no feasible LR or all feasible points have
# weak efficacy.
EXPANDED_LR_GRID = {
    ("Phi-3", "openbook"): [1e-5, 3e-5, 5e-5, 1e-4, 3e-4, 5e-4],
    ("Phi-3", "sqa"): [5e-6, 1e-5, 3e-5, 5e-5, 1e-4, 3e-4],
    ("LLaMA-3-3B", "openbook"): [3e-6, 5e-6, 1e-5, 3e-5, 5e-5, 1e-4],
    ("LLaMA-3-3B", "sqa"): [3e-6, 5e-6, 1e-5, 3e-5, 5e-5, 1e-4],
}

GRID = COMPACT_LR_GRID

## Planned Runs

In [ ]:
def iter_runs(grid, short_model=None, dataset=None, lr_values=None):
    for (model_name, dataset_name), configured_lrs in grid.items():
        if short_model is not None and model_name != short_model:
            continue
        if dataset is not None and dataset_name != dataset:
            continue
        lrs = lr_values if lr_values is not None else configured_lrs
        for lr in lrs:
            yield model_name, dataset_name, lr


planned_runs = list(iter_runs(GRID))
for short_model, dataset, lr in planned_runs:
    print(f"{short_model}\t{dataset}\tlr={lr}")
print(f"Total planned runs: {len(planned_runs)}")

## Run LMF+KL Pilot Points

Set one of the run toggles below to `True` when you want to launch training. Keeping the default `False` makes the notebook safe to run top-to-bottom for analysis only.

In [ ]:
def run_lmf_pilot_point(
    short_model,
    dataset,
    lr,
    results_dir=RESULTS_DIR,
    n_unlearn=N_UNLEARN,
    n_verify=N_VERIFY,
    epochs=EPOCHS,
    rt_lambda=RT_LAMBDA,
):
    from repro import config as cfg
    from repro.run_repro import build_args, patched_main

    old_method = cfg.METHOD
    cfg.METHOD = METHOD
    log_suffix = "" if rt_lambda == 1.0 else f"_lambda={rt_lambda:g}"
    try:
        args = build_args(
            short_model=short_model,
            dataset=dataset,
            lr=lr,
            smoke=False,
            rt_lambda=rt_lambda,
            results_dir=results_dir,
            n_unlearn=n_unlearn,
            n_verify=n_verify,
            epochs=epochs,
            log_suffix=log_suffix,
        )
        args.method = METHOD
        patched_main(args)
    finally:
        cfg.METHOD = old_method


# Single pilot point. Edit this tuple, set RUN_SINGLE=True, then run this cell.
RUN_SINGLE = False
SINGLE_RUN = ("Phi-3", "openbook", 3e-5)

if RUN_SINGLE:
    run_lmf_pilot_point(*SINGLE_RUN)

In [ ]:
# Full grid. Set RUN_GRID=True only when you are ready for a long GPU run.
RUN_GRID = True

if RUN_GRID:
    for short_model, dataset, lr in iter_runs(GRID):
        print("=" * 72)
        print(f"LMF+KL pilot: {short_model} x {dataset} lr={lr}")
        print("=" * 72)
        run_lmf_pilot_point(short_model, dataset, lr)

## Load Results and Compute Appendix-E Metrics

In [ ]:
def result_path(results_dir, dataset, short_model, lr, rt_lambda=RT_LAMBDA):
    lambda_suffix = "" if rt_lambda == 1.0 else f"_lambda={rt_lambda:g}"
    filename = (
        f"{METHOD}_sentencize_s=True_lr={lr}{lambda_suffix}"
        f"_rs=1001_pos=True_ff2=True.out"
    )
    return ROOT / results_dir / dataset / short_model / filename


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as infile:
        return [json.loads(line) for line in infile]


def mean(values):
    return sum(values) / len(values) if values else 0.0


def argmax(values):
    return max(range(len(values)), key=lambda idx: values[idx])


def changed_prediction(results):
    unique_questions = {result["question"] for result in results}
    flipped_questions = set()
    for result in results:
        epoch_results = result["unlearning_results"]
        preds = [argmax(epoch["probs"]) for _, epoch in epoch_results.items()]
        if any(pred != preds[0] for pred in preds):
            flipped_questions.add(result["question"])
    return len(flipped_questions) / len(unique_questions) * 100.0


def compute_specificity(results):
    specificity_by_epoch = {}
    for result in results:
        epoch_results = result["unlearning_results"]
        initial_preds = epoch_results["0"]["specificity_preds"]
        for epoch_idx in range(1, len(epoch_results)):
            preds = epoch_results[str(epoch_idx)]["specificity_preds"]
            same = sum(1 for before, after in zip(initial_preds, preds) if before == after)
            specificity_by_epoch.setdefault(epoch_idx - 1, []).append(same / len(preds) * 100.0)

    all_specificity = [
        score
        for epoch_scores in specificity_by_epoch.values()
        for score in epoch_scores
    ]
    return mean(all_specificity)


def average_efficacy(results):
    efficacy_by_epoch = {}
    for result in results:
        epoch_results = result["unlearning_results"]
        probabilities = [
            math.exp(epoch["cot_step_prob"][0])
            for _, epoch in epoch_results.items()
        ]
        p_0 = probabilities[0]
        for epoch_idx, p_i in enumerate(probabilities):
            efficacy_by_epoch.setdefault(epoch_idx, [])
            if efficacy_by_epoch[epoch_idx]:
                efficacy_by_epoch[epoch_idx].append((1 - p_i / p_0) * 100.0)
            else:
                efficacy_by_epoch[epoch_idx].append(0.0)

    all_efficacy = [
        score
        for epoch_scores in efficacy_by_epoch.values()
        for score in epoch_scores
    ]
    return mean(all_efficacy)


def make_stats(results):
    return {
        "n_instances": len({result["question"] for result in results}),
        "faithfulness": changed_prediction(results),
        "efficacy": average_efficacy(results),
        "specificity": compute_specificity(results),
        "n_cot_steps": len(results),
    }

In [ ]:
def summarize_grid(grid=GRID, results_dir=RESULTS_DIR, rt_lambda=RT_LAMBDA):
    rows = []
    for short_model, dataset, lr in iter_runs(grid):
        path = result_path(results_dir, dataset, short_model, lr, rt_lambda)
        if not path.exists():
            rows.append({
                "dataset": dataset,
                "model": short_model,
                "lr": lr,
                "status": "missing",
                "path": str(path),
            })
            continue
        stats = make_stats(load_jsonl(path))
        rows.append({
            "dataset": dataset,
            "model": short_model,
            "lr": lr,
            "status": "ok",
            "path": str(path),
            **stats,
        })
    return rows


rows = summarize_grid()
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for row in rows:
        print(row)

## Select Best Learning Rates

Criterion: maximize efficacy subject to `round(specificity) >= 95`. This matches Appendix E. Faithfulness is printed only for context.

In [ ]:
def select_best_lr(rows, threshold=SPECIFICITY_THRESHOLD):
    grouped = {}
    for row in rows:
        if row["status"] != "ok":
            continue
        grouped.setdefault((row["dataset"], row["model"]), []).append(row)

    selected = []
    for (dataset, model), candidates in sorted(grouped.items()):
        feasible = [row for row in candidates if round(row["specificity"]) >= threshold]
        if feasible:
            best = max(feasible, key=lambda row: row["efficacy"])
            selected.append({
                "dataset": dataset,
                "model": model,
                "best_lr": best["lr"],
                "efficacy": best["efficacy"],
                "specificity": best["specificity"],
                "faithfulness": best["faithfulness"],
                "criterion": f"max E subject to round(S)>={threshold}",
            })
        else:
            best = max(candidates, key=lambda row: row["specificity"])
            selected.append({
                "dataset": dataset,
                "model": model,
                "best_lr": None,
                "best_specificity_lr": best["lr"],
                "efficacy": best["efficacy"],
                "specificity": best["specificity"],
                "faithfulness": best["faithfulness"],
                "criterion": "no feasible LR; expand/lower grid",
            })
    return selected


selected = select_best_lr(rows)
try:
    import pandas as pd
    display(pd.DataFrame(selected))
except ImportError:
    for row in selected:
        print(row)

## If No Learning Rate Is Feasible

If a cell has no LR with `round(specificity) >= 95`, rerun that cell with `GRID = EXPANDED_LR_GRID`, or manually test lower LR values. For LMF+KL, loss strength can differ substantially from NPO+KL, so lower LRs are often the first thing to try when specificity is below the threshold.